In [3]:
! pip3 install transformers
! pip3 install datasets
! pip3 install scipy sklearn
! pip3 install huggingface_hub

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 679.2 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.0/575.0 kB 2.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from datasets import load_dataset, load_from_disk
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import ModelCheckpoint
from transformers import AutoTokenizer
from transformers import create_optimizer
from transformers import TFAutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers.keras_callbacks import PushToHubCallback
from huggingface_hub import notebook_login

In [5]:
training_data_file = "dataset/train.csv"
test_data_fine = "dataset/test.csv"
model_checkpoint = "distilbert-base-uncased"
batch_size = 16

In [6]:
dataset = load_dataset('csv', data_files = [training_data_file])
dataset = dataset['train'].train_test_split(test_size=0.1)
dataset['valid'] = dataset['test']
dataset['test'] = load_dataset('csv', data_files = [test_data_fine])['train']

Using custom data configuration default-338415cb58fbd2c5


Extracting data files: 100%|██████████| 1/1 [00:00<00:00, 134.01it/s]


Dataset csv downloaded and prepared to /Users/wayne/.cache/huggingface/datasets/csv/default-338415cb58fbd2c5/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519. Subsequent calls will reuse this data.


100%|██████████| 1/1 [00:00<00:00, 202.67it/s]
Using custom data configuration default-1d42ed1cb3e3f366


Extracting data files: 100%|██████████| 1/1 [00:00<00:00, 466.76it/s]


Dataset csv downloaded and prepared to /Users/wayne/.cache/huggingface/datasets/csv/default-1d42ed1cb3e3f366/0.0.0/433e0ccc46f9880962cc2b12065189766fbb2bee57a221866138fb9203c83519. Subsequent calls will reuse this data.


100%|██████████| 1/1 [00:00<00:00, 459.10it/s]


In [7]:
pd.DataFrame(dataset['train'])

,id,keyword,location,text,target
0,6478,injured,None,#golf McIlroy fuels PGA speculation after vide...,0
1,1015,blazing,Everywhere,**Let - Me - Be - Your - Hot - Blazing - Fanta...,0
2,2463,collided,None,I scored 111020 points in PUNCH QUEST stopped ...,1
3,8562,screams,#Gladiator Û¢860Û¢757Û¢,Casually on the phone with Jasmine while she c...,0
4,5212,fatality,playing soccer & eating pizza,@Jake_ADavis @FaTality_US we are cuddling righ...,0
...,...,...,...,...,...
6846,4243,drowned,India,Hundreds of migrants feared drowned off Libya:...,1
6847,5095,famine,Ireland,I've experienced the smell of rotting potatoes...,1
6848,2568,crash,"21.462446,-158.022017",The Next Financial Crash. 'The Writing is on t...,0
6849,149,aftershock,304,'The man who can drive himself further once th...,0


In [8]:
pd.DataFrame(dataset['valid'])

,id,keyword,location,text,target
0,9991,tsunami,in the Word of God,@author_mike Amen today is the Day of Salvatio...,1
1,9005,stretcher,U.S.A and Canada,organic natural horn stretcher expander earrin...,0
2,1980,bush%20fires,Trinidad and Tobago,Drought fuels bush fires in Jamaica - http://t...,1
3,6804,loud%20bang,Kenya,Ercjmnea: Breaking news! Unconfirmed! I just h...,0
4,8970,storm,None,A Warcraft 3-inspired mode is likely coming to...,0
...,...,...,...,...,...
757,2485,collided,bk.,She looked back &amp; her daughter &amp; said ...,0
758,7288,nuclear%20disaster,US,3 Former Executives to Be Prosecuted in Fukush...,1
759,7733,panicking,VCU,Just realized that maybe it not normal to sit ...,0
760,7590,outbreak,None,An outbreak of Legionnaires' disease in New Yo...,1


In [9]:
pd.DataFrame(dataset['test'])

,id,keyword,location,text
0,0,None,None,Just happened a terrible car crash
1,2,None,None,"Heard about #earthquake is different cities, s..."
2,3,None,None,"there is a forest fire at spot pond, geese are..."
3,9,None,None,Apocalypse lighting. #Spokane #wildfires
4,11,None,None,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,None,None,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,None,None,Storm in RI worse than last hurricane. My city...
3260,10868,None,None,Green Line derailment in Chicago http://t.co/U...
3261,10874,None,None,MEG issues Hazardous Weather Outlook (HWO) htt...


In [10]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

pre_tokenizer_columns = set(dataset["train"].features)
encoded_dataset = dataset.map(tokenize_function, batched=True)
tokenizer_columns = list(set(encoded_dataset["train"].features) - pre_tokenizer_columns)
print("Columns added by tokenizer:", tokenizer_columns)

100%|██████████| 1/1 [00:00<00:00,  5.65ba/s]

Columns added by tokenizer: ['input_ids', 'attention_mask']


In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target'],
        num_rows: 762
    })
})

In [12]:
encoded_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 6851
    })
    test: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'input_ids', 'attention_mask'],
        num_rows: 3263
    })
    valid: Dataset({
        features: ['id', 'keyword', 'location', 'text', 'target', 'input_ids', 'attention_mask'],
        num_rows: 762
    })
})

In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

tf_train_dataset = encoded_dataset['train'].to_tf_dataset(
    columns=tokenizer_columns,
    label_cols=["target"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=batch_size,
)

tf_validation_dataset = encoded_dataset['valid'].to_tf_dataset(
    columns=tokenizer_columns,
    label_cols=["target"],
    shuffle=False,
    batch_size=batch_size,
    collate_fn=data_collator,
)

tf_test_dataset = encoded_dataset['test'].to_tf_dataset(
    columns=tokenizer_columns,
    shuffle=False,
    batch_size=batch_size,
    collate_fn=data_collator,
)

tf_test_dataset

<PrefetchDataset element_spec={'input_ids': TensorSpec(shape=(None, None), dtype=tf.int64, name=None), 'attention_mask': TensorSpec(shape=(None, None), dtype=tf.int64, name=None)}>

In [14]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=2
)

2022-05-31 16:14:46.331716: W tensorflow/python/util/util.cc:368] Sets are not currently considered sequences, but this may change in the future, so consider avoiding using them.
Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertForSequenceClassification: ['activation_13', 'vocab_layer_norm', 'vocab_projector', 'vocab_transform']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFDistilBertForSequenceClassification were not initialized from the model checkpoint 

In [15]:
num_epochs = 3
batches_per_epoch = len(encoded_dataset["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)

optimizer, schedule = create_optimizer(
    init_lr=2e-5, num_warmup_steps=0, num_train_steps=total_train_steps
)
loss = loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

In [16]:
model.fit(
    tf_train_dataset,
    validation_data=tf_validation_dataset,
    epochs=3,
#     callbacks=callbacks,
)

Epoch 1/3
 11/428 [..............................] - ETA: 1:51:36 - loss: 0.6871 - accuracy: 0.5227

In [ ]:
test_pred = model.predict(tf_test_dataset)

In [ ]:
submission = pd.read_csv('../input/nlp-getting-started/sample_submission.csv')
submission['target'] = np.argmax(test_pred.logits, axis=1)
submission.to_csv('submission.csv', index=False)